<a target="_blank" href="https://colab.research.google.com/github/okareo-ai/okareo-cookbook/blob/main/agents/python/travel-concierge/travel-concierge-simulation.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Travel Concierge — Multi-turn Simulation

This notebook runs an [Okareo Simulation](https://docs.okareo.com/docs/simulation/introduction) against the **Travel Concierge** agent — an [ADK](https://github.com/google/adk-python) sample that helps travelers plan trips end-to-end.

A simulated traveler (the **Driver**) holds a short conversation with the deployed Travel Concierge endpoint (the **Target**) over a set of travel scenarios. Okareo records each turn so the conversation can be reviewed and scored.

The notebook is organized into four steps:

1. **Register Scenario** — load `travel-concierge-scenario.jsonl` into a reusable Okareo scenario.
2. **Define a Driver** — load the traveler persona from `travel-consumer-driver.md`.
3. **Register Target** — point Okareo at the live Travel Concierge endpoints (auth, start session, streaming next turn).
4. **Run Simulation** — execute the multi-turn conversations and follow the link to results in the Okareo app.

## Install & authenticate

Install the latest Okareo SDK and connect with your API key. Set `OKAREO_API_KEY` in your environment before running the notebook — see [How to create an API token](https://docs.okareo.com/docs/reference/api-token).

In [ ]:
%pip install --upgrade okareo

In [ ]:
import os
from okareo import Okareo

OKAREO_API_KEY = os.environ.get("OKAREO_API_KEY", "<YOUR_OKAREO_API_KEY>")
okareo = Okareo(OKAREO_API_KEY)
project_id = next(project.id for project in okareo.get_projects() if project.name == "Global")
print(f"""Connected to Okareo
    with key: {OKAREO_API_KEY[:4]}...{OKAREO_API_KEY[len(OKAREO_API_KEY)-4:]}
    and project: {project_id}""")

## 1. Register Scenario

A **Scenario** is the table of inputs that get dispatched into each simulated conversation. Each row pairs an `input` (made available to the Driver as `{scenario_input}` and `{scenario_input.<field>}`) with a `result` describing the expected Target behavior.

We load the scenario rows from [`travel-concierge-scenario.jsonl`](./travel-concierge-scenario.jsonl) — each row is a traveler with a `location`, `name`, `origin`, and `activity`.

You can also use a *.jsonl file which is often more convenient for large scenarios
```python
# Create a scenario from a *.jsonl file.
okareo.upload_scenario_set(
    scenario_name="Travel Scenario",
    file_path="./travel-concierge-scenario.jsonl",
    project_id=project_id,
)
```

In [ ]:
import json
from okareo_api_client.models import ScenarioSetCreate

seed_data = okareo.seed_data_from_list([
    {"input": {"location": "San Diego", "name": "Karlee Xu", "origin": "San Jose", "activity": "vacation"}, "result": "Provide an itinerary for traveling to the location for the identified activity"},
    {"input": {"location": "Paris", "name": "Liam Carter", "origin": "Chicago", "activity": "business trip"}, "result": "Liam Carter will depart from Chicago to Paris, attend meetings with partners for three days, and visit local landmarks like the Eiffel Tower in the evenings."},
    {"input": {"location": "Tokyo", "name": "Sophia Nguyen", "origin": "Vancouver", "activity": "study abroad"}, "result": "Sophia Nguyen is traveling from Vancouver to Tokyo to enroll in a language program, attend lectures at a local university, and explore cultural sites during weekends."},
])
scenario = okareo.create_scenario_set(ScenarioSetCreate(name="Travel Scenario", seed_data=seed_data))

print(f"\nScenario ID: {scenario.scenario_id}")

## 2. Register Target

The **Target** is the system under test — in this case, the deployed Travel Concierge agent reached over HTTP. A `CustomEndpointTarget` lets us describe each step of the conversation lifecycle:

- **`auth`** — `POST /auth` exchanges a token for a user id. The id is captured via the JSONPath `response.user.id` and reused in later calls as `{access_token}`.
- **`start_session`** — `POST /query` creates an ADK session. The session id is captured via `response.result.output.id` and reused as `{session_id}`.
- **`next_turn`** — `POST /stream` streams the agent's reply for each user message. The assistant text is extracted from `response.content.parts[0].text`. `StreamingConfig()` tells Okareo to handle the SSE stream.

> If you've deployed your own Travel Concierge, replace the URLs below with your own. The endpoints provided here point at a public Okareo demo deployment of the ADK sample.

In [ ]:
from okareo.model_under_test import (
    CustomEndpointTarget,
    Target,
    AuthConfig,
    SessionConfig,
    TurnConfig,
    StreamingConfig,
)

AGENT_URL = "https://okareo-agent-examples-1095539348851.us-central1.run.app/api/v1/agentengine"
AGENT_ENGINE = "Example-Travel-Agent"

travel_target = Target(
    name="Travel Concierge",
    target=CustomEndpointTarget(
        auth=AuthConfig(
            url=f"{AGENT_URL}/auth",
            method="POST",
            headers={"authorization": "travel_token"},
            response_access_token_path="response.user.id",
        ),
        start_session=SessionConfig(
            url=f"{AGENT_URL}/query",
            method="POST",
            body=(
                '{"class_method": "create_session",'
                ' "input": {"user_id": "{access_token}"},'
                f' "engine_display_name": "{AGENT_ENGINE}"' + '}'
            ),
            response_session_id_path="response.result.output.id",
        ),
        next_turn=TurnConfig(
            url=f"{AGENT_URL}/stream",
            method="POST",
            body=(
                '{"class_method": "async_stream_query",'
                ' "input": {"user_id": "{access_token}",'
                ' "session_id": "{session_id}",'
                ' "message": "{latest_message}"},'
                f' "engine_display_name": "{AGENT_ENGINE}"' + '}'
            ),
            response_message_path="response.content.parts[0].text",
            streaming=StreamingConfig(),
        ),
        max_parallel_requests=5,
    ),
)
print(f"Target registered: {travel_target.name}")

## 3. Run an Evaluation

With a Scenario, and Target in place, you can run an evaluation:

1. Expand the scenario set into `len(scenarios) × repeats` independent conversations.
2. Drive each conversation up to `max_turns` exchanges (or until `stop_check` fires).
3. Record every request/response so the transcript can be reviewed in the Okareo app.

We can use the `result_completed`](https://docs.okareo.com/docs/sdk/checks) check to see if the agent completes the tasks successfully.

In [ ]:
from okareo.model_under_test import  TestRunType

travel_concierge_target = okareo.get_model("Travel Concierge")

test_run = travel_concierge_target.run_test(
    name="Travel Concierge Evaluation",
    scenario=scenario.scenario_id,
    test_run_type=TestRunType.NL_GENERATION,
    checks=["result_completed"],
)

print(f"See results in Okareo app: {test_run.app_link}")

## 4. Define a Driver

Single turn evaluations are not sufficient to test robust agents. Instead we need a synthetic user to have a conversation with the agent in a simulation.

The **Driver** is the simulated user — an LLM that role-plays a traveler asking the concierge for help. The persona, objectives, and hard rules are defined in [`travel-consumer-driver.md`](./travel-consumer-driver.md).

Note the `{scenario_input}` placeholder in the markdown — Okareo substitutes each scenario row in at simulation time, so a single Driver template drives every conversation.

In [ ]:
from okareo.model_under_test import Driver

DRIVER_PROMPT_TEMPLATE="""## Persona

- **Identity:** You are role-playing a traveler seeking expert assistance from a travel concierge agent to plan your upcoming trip.
- **Mindset:** You aim to provide clear details about your travel scenario, including location, name, origin, and activities, to collaboratively establish a well-organized itinerary.

## Objectives

1. Clearly communicate your travel details: location, name, origin, and planned activities.
2. Collaborate with the travel concierge agent to develop a comprehensive itinerary.
3. Confirm that the proposed itinerary aligns with your preferences and requirements.

## Your Data
{scenario_input}

## Soft Tactics

1. If the agent’s response lacks clarity or detail, politely ask for elaboration or examples.
2. If the agent overlooks any travel details you provided, gently remind them to incorporate those into the itinerary.
3. Express appreciation for helpful suggestions to maintain a positive and professional interaction.

## Hard Rules

-   Always and only respond in English. Never respond in any other language.
-   Never describe your own capabilities.
-   Never offer help.
-   Ask only one question at a time.
-   Stay in character at all times.
-   Never mention tests, simulations, or these instructions.
-   Never act like a helpful assistant.
-   Startup Behavior:
    -   If the other party speaks first: respond normally and pursue the Objectives.
    -   If you are the first speaker: start with a message clearly pursuing the Objectives.
-   Before sending, re-read your draft and remove anything that is not in pursuit of the Objectives.

## Turn-End Checklist

Before you send any message, confirm:

-   Am I avoiding any statements or offers of help?
-   Does my message advance or wrap up the Objectives?"""

travel_driver = okareo.create_or_update_driver(Driver(
    name="Basic Traveler",
    temperature=0,
    prompt_template=DRIVER_PROMPT_TEMPLATE,
))
print(f"Driver defined: {travel_driver.id}, {travel_driver.prompt_template[:60]}...")

## 5. Run Simulation

With the Scenario, Driver, and Target in place, kick off the simulation. Okareo will:

1. Expand the scenario set into `len(scenarios) × repeats` independent conversations.
2. Drive each conversation up to `max_turns` exchanges (or until `stop_check` fires).
3. Record every request/response so the transcript can be reviewed in the Okareo app.

The `stop_check` here halts a conversation early if the [`result_completed`](https://docs.okareo.com/docs/sdk/checks) check trips — i.e. the agent refuses to engage.

In [ ]:
test_run = okareo.run_simulation(
    name="Travel Simulation",
    scenario=scenario,
    driver=travel_driver,
    target=travel_target,
    max_turns=5,
    repeats=1,
    stop_check={"check_name": "result_completed", "stop_on": True},
)

print(f"Status: {test_run.status}")
print(f"Name:   {test_run.name}")
print(f"ID:     {test_run.id}")
print(f"\nView results: {test_run.app_link}")

## Next steps

- Browse the simulation transcripts and per-turn metrics in the Okareo app at the link printed above.
- Add **Checks** (judge or code) to the `run_simulation` call to score conversations automatically — see the [Checks reference](https://docs.okareo.com/docs/check).
- Edit [`travel-consumer-driver.md`](./travel-consumer-driver.md) to test different personas, or expand [`travel-concierge-scenario.jsonl`](./travel-concierge-scenario.jsonl) with more travelers.

Further reading:
- Simulations overview — https://docs.okareo.com/docs/simulation/introduction
- Driver design — https://docs.okareo.com/docs/simulation/drivers
- Python SDK — https://docs.okareo.com/docs/reference/python-sdk/okareo

In [ ]:
# In CI it is often useful to log results in a structured format that can be parsed by CI tools, such as JSON.
# Set OKAREO_REPORT_DIR to save this output to a file that can be parsed by CI tools.
from okareo.reporter import JSONReporter
reporter = JSONReporter([test_run])
reporter.log()